In [1]:
import numpy as np


def cleanup():
    import gc
    import torch
    torch.mps.empty_cache()
    gc.collect()

In [2]:
import torch
import torch.nn as nn
import math

# 1. Squeeze-and-Excitation Block
# This helps the model focus on the most important parts of the image
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduced_dim):
        super(SEBlock, self).__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, reduced_dim, 1),
            nn.SiLU(), # EfficientNet uses SiLU (Swish) instead of ReLU
            nn.Conv2d(reduced_dim, in_channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.se(x)

# 2. MBConv Block (The heart of EfficientNet)
class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, expand_ratio, se_ratio=0.25):
        super(MBConv, self).__init__()
        expanded_channels = in_channels * expand_ratio
        self.use_residual = (stride == 1 and in_channels == out_channels)

        layers = []
        # Expansion phase
        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, expanded_channels, 1, bias=False),
                nn.BatchNorm2d(expanded_channels),
                nn.SiLU()
            ])

        # Depthwise phase
        layers.extend([
            nn.Conv2d(expanded_channels, expanded_channels, kernel_size, stride, kernel_size//2, groups=expanded_channels, bias=False),
            nn.BatchNorm2d(expanded_channels),
            nn.SiLU()
        ])

        # Squeeze-and-Excitation
        layers.append(SEBlock(expanded_channels, int(in_channels * se_ratio)))

        # Output phase
        layers.extend([
            nn.Conv2d(expanded_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        ])

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_residual:
            return x + self.block(x)
        return self.block(x)

# 3. EfficientNet-B0 Main Class
class EfficientNetB0(nn.Module):
    def __init__(self, num_classes=10):
        super(EfficientNetB0, self).__init__()

        # Modified initial layer: stride=1 instead of stride=2 for 32x32 images
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU()
        )

        # EfficientNet-B0 Config: [expand_ratio, channels, repeats, stride, kernel_size]
        self.config = [
            [1, 16, 1, 1, 3],
            [6, 24, 2, 2, 3],
            [6, 40, 2, 2, 5],
            [6, 80, 3, 2, 3],
            [6, 112, 3, 1, 5],
            [6, 192, 4, 2, 5],
            [6, 320, 1, 1, 3],
        ]

        layers = []
        in_channels = 32
        for expand_ratio, out_channels, repeats, stride, kernel_size in self.config:
            for i in range(repeats):
                layers.append(MBConv(in_channels, out_channels, kernel_size, stride if i == 0 else 1, expand_ratio))
                in_channels = out_channels
        self.layers = nn.Sequential(*layers)

        self.head = nn.Sequential(
            nn.Conv2d(in_channels, 1280, 1, bias=False),
            nn.BatchNorm2d(1280),
            nn.SiLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(1280, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layers(x)
        x = self.head(x)
        return x

# Initialize and move to M1 GPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = EfficientNetB0(num_classes=10).to(device)

print(f"EfficientNet-B0 scratch built. Total layers in sequence: {len(list(model.children()))}")

EfficientNet-B0 scratch built. Total layers in sequence: 3


In [3]:
loss_fn = nn.CrossEntropyLoss()

In [4]:
optimiser = torch.optim.Adam(model.parameters(),lr=0.0001)

In [5]:
from torchvision import transforms
import torchvision

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

In [6]:
shared_root='/Users/benjaminbrooke/PycharmProjects/Python_PyTroch/IT 599 Research Paper/1.LeNet-5/data/'

train_set = torchvision.datasets.CIFAR10(root=shared_root, train=True, download=True, transform=transform)

In [7]:
print(next(iter(train_set)))

(tensor([[[-0.5373, -0.6627, -0.6078,  ...,  0.2392,  0.1922,  0.1608],
         [-0.8745, -1.0000, -0.8588,  ..., -0.0353, -0.0667, -0.0431],
         [-0.8039, -0.8745, -0.6157,  ..., -0.0745, -0.0588, -0.1451],
         ...,
         [ 0.6314,  0.5765,  0.5529,  ...,  0.2549, -0.5608, -0.5843],
         [ 0.4118,  0.3569,  0.4588,  ...,  0.4431, -0.2392, -0.3490],
         [ 0.3882,  0.3176,  0.4039,  ...,  0.6941,  0.1843, -0.0353]],

        [[-0.5137, -0.6392, -0.6235,  ...,  0.0353, -0.0196, -0.0275],
         [-0.8431, -1.0000, -0.9373,  ..., -0.3098, -0.3490, -0.3176],
         [-0.8118, -0.9451, -0.7882,  ..., -0.3412, -0.3412, -0.4275],
         ...,
         [ 0.3333,  0.2000,  0.2627,  ...,  0.0431, -0.7569, -0.7333],
         [ 0.0902, -0.0353,  0.1294,  ...,  0.1608, -0.5137, -0.5843],
         [ 0.1294,  0.0118,  0.1137,  ...,  0.4431, -0.0745, -0.2784]],

        [[-0.5059, -0.6471, -0.6627,  ..., -0.1529, -0.2000, -0.1922],
         [-0.8431, -1.0000, -1.0000,  ..., -

In [8]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_set,shuffle= True, batch_size=50)

In [10]:
from tqdm.notebook import tqdm
import time
from torch.utils.tensorboard import SummaryWriter
LeNet5_Metrics = SummaryWriter()

latency_per_image = []
global_i = 0

all_preds = []
all_labels = []

for image,label in train_loader:

    image = image.to(device)
    label = label.to(device)

    start_time = time.time()

    optimiser.zero_grad()

    y_pred =  model(image)

    loss = loss_fn(y_pred,label)

    loss.backward()

    optimiser.step()

    _, predicted = torch.max(y_pred.data, 1)
    correct = (predicted == label).sum().item()
    accuracy = correct / label.size(0)

    end_time = time.time()

    latency_per_image.append(end_time - start_time)

    LeNet5_Metrics.add_scalar("Loss/train - EfficientNet-B0 IT599 Ben", loss.item(), global_i)
    LeNet5_Metrics.add_scalar("Accuracy/train - EfficientNet-B0 IT599 Ben", accuracy, global_i)

    all_preds.extend(predicted.cpu())
    all_labels.extend(label.cpu())

    global_i += 1

    print(global_i/len(train_loader))


LeNet5_Metrics.close()

avg_latency_batch = sum(latency_per_image) / len(latency_per_image)
avg_latency_image = avg_latency_batch / train_loader.batch_size
throughput = 1 / avg_latency_image

print(f"Latency: {avg_latency_image*1000:.2f} ms | Throughput: {throughput:.2f} items/sec")

0.001
0.002
0.003
0.004
0.005
0.006
0.007
0.008
0.009
0.01
0.011
0.012
0.013
0.014
0.015
0.016
0.017
0.018
0.019
0.02
0.021
0.022
0.023
0.024
0.025
0.026
0.027
0.028
0.029
0.03
0.031
0.032
0.033
0.034
0.035
0.036
0.037
0.038
0.039
0.04
0.041
0.042
0.043
0.044
0.045
0.046
0.047
0.048
0.049
0.05
0.051
0.052
0.053
0.054
0.055
0.056
0.057
0.058
0.059
0.06
0.061
0.062
0.063
0.064
0.065
0.066
0.067
0.068
0.069
0.07
0.071
0.072
0.073
0.074
0.075
0.076
0.077
0.078
0.079
0.08
0.081
0.082
0.083
0.084
0.085
0.086
0.087
0.088
0.089
0.09
0.091
0.092
0.093
0.094
0.095
0.096
0.097
0.098
0.099
0.1
0.101
0.102
0.103
0.104
0.105
0.106
0.107
0.108
0.109
0.11
0.111
0.112
0.113
0.114
0.115
0.116
0.117
0.118
0.119
0.12
0.121
0.122
0.123
0.124
0.125
0.126
0.127
0.128
0.129
0.13
0.131
0.132
0.133
0.134
0.135
0.136
0.137
0.138
0.139
0.14
0.141
0.142
0.143
0.144
0.145
0.146
0.147
0.148
0.149
0.15
0.151
0.152
0.153
0.154
0.155
0.156
0.157
0.158
0.159
0.16
0.161
0.162
0.163
0.164
0.165
0.166
0.167
0.168
0.169
0.1

In [ ]:
#python3 -m tensorboard.main --logdir="/Users/benjaminbrooke/PycharmProjects/Python_PyTroch/IT 599 Research Paper/5.EfficientNet-B0/runs"